# Tăng cường đặc trưng VSL

Tạo bản nhiễu từ mẫu gốc trong `data/processed`; `data/raw` giữ nguyên. Đặt `MIN_SAMPLES_PER_CLASS = N` để chỉ bổ sung các lớp có ít hơn N mẫu hoàn chỉnh trong `processed`, tạo đúng số còn thiếu. Đặt `-1` để tăng cường tất cả mẫu gốc theo `COPIES_PER_SAMPLE` mà không giới hạn. Mỗi bản mới ghi `origin_sample_id` để đi cùng fold với mẫu gốc. Validation trong `train.ipynb` chỉ chấm mẫu gốc.


In [9]:
import importlib, sys
from pathlib import Path
import numpy as np
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None: raise FileNotFoundError('Chạy Jupyter từ dự án hoặc notebooks/')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
for name in [m for m in sys.modules if m == 'src' or m.startswith('src.')]: del sys.modules[name]
reader = importlib.import_module('src.data.data_reader')
augment = importlib.import_module('src.data.augment')
DataHandler = reader.DataHandler
PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw'
SELECTED_CLASSES = 'all'
MIN_SAMPLES_PER_CLASS = int(input("MIN_SAMPLES_PER_CLASS (-1 to all): "))
COPIES_PER_SAMPLE = 2
NOISE_STRENGTH = 1.0
PITCH_DEGREES = augment.PITCH_AUG_DEG
SEED = None
if MIN_SAMPLES_PER_CLASS < -1 or COPIES_PER_SAMPLE < 1 or not 0 < NOISE_STRENGTH <= 3: raise ValueError('Cấu hình ngưỡng/số bản/noise không hợp lệ')
if not 0 <= PITCH_DEGREES <= 45: raise ValueError('PITCH_DEGREES phải trong [0, 45]')
classes = None if SELECTED_CLASSES == 'all' else ([SELECTED_CLASSES] if isinstance(SELECTED_CLASSES, str) else SELECTED_CLASSES)
plan, counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
print(f'{sum(copies for _, copies in plan)} mẫu dự kiến tạo trong {len(counts)} lớp')

25 mẫu dự kiến tạo trong 31 lớp


## Kiểm tra trước khi tạo


In [10]:
for name, count in counts.items():
    extra = sum(copies for source, copies in plan if source.parent.name == name)
    print(f'{name}: hiện có {count}, thêm {extra}' + (f', sau đó có {count + extra}' if MIN_SAMPLES_PER_CLASS != -1 else ''))
print('Đã đủ ngưỡng, không cần tạo thêm.' if not plan else 'Raw và mẫu gốc được giữ nguyên. Chạy ô tiếp theo để tạo bản nhiễu.')


a: hiện có 45, thêm 0, sau đó có 45
age: hiện có 45, thêm 0, sau đó có 45
b: hiện có 45, thêm 0, sau đó có 45
c: hiện có 45, thêm 0, sau đó có 45
cảm ơn: hiện có 45, thêm 0, sau đó có 45
dấu huyền: hiện có 45, thêm 0, sau đó có 45
dấu hỏi: hiện có 45, thêm 0, sau đó có 45
dấu nặng: hiện có 45, thêm 0, sau đó có 45
dấu sắc: hiện có 45, thêm 0, sau đó có 45
dấu â,ô,ê: hiện có 45, thêm 0, sau đó có 45
dấu ă: hiện có 45, thêm 0, sau đó có 45
dấu ơ,ư: hiện có 45, thêm 0, sau đó có 45
e: hiện có 45, thêm 0, sau đó có 45
g: hiện có 45, thêm 0, sau đó có 45
gặp: hiện có 20, thêm 25, sau đó có 45
h: hiện có 45, thêm 0, sau đó có 45
i: hiện có 45, thêm 0, sau đó có 45
k: hiện có 45, thêm 0, sau đó có 45
l: hiện có 45, thêm 0, sau đó có 45
m: hiện có 45, thêm 0, sau đó có 45
n: hiện có 45, thêm 0, sau đó có 45
o: hiện có 45, thêm 0, sau đó có 45
r: hiện có 45, thêm 0, sau đó có 45
t: hiện có 45, thêm 0, sau đó có 45
ten_la: hiện có 45, thêm 0, sau đó có 45
toi: hiện có 45, thêm 0, sau đó có 45
u:

## Tạo mẫu tăng cường
Ô này tính lại số mẫu hiện có ngay trước khi ghi. Với ngưỡng N, chạy lại sau khi đã đủ N sẽ không tạo thêm; với N = -1, mỗi lần chạy tạo thêm bản mới.


In [11]:
rng = np.random.default_rng(SEED)
run_plan, current_counts = augment.plan_augmentation(PROCESSED_ROOT, classes, MIN_SAMPLES_PER_CLASS, COPIES_PER_SAMPLE)
created, errors = [], []
for index, (source, copies) in enumerate(run_plan, 1):
    try:
        paths = augment.augment_sample(source, RAW_ROOT, copies, NOISE_STRENGTH, rng, pitch_degrees=PITCH_DEGREES)
        created.extend(paths)
        print(f'[{index}/{len(run_plan)}] {source.parent.name}/{source.name} -> {", ".join(path.name for path in paths)}', flush=True)
    except Exception as exc:
        errors.append((str(source), str(exc)))
        print(f'[{index}/{len(run_plan)}] LỖI {source}: {exc}', flush=True)
print(f'Đã tạo {len(created)} mẫu; lỗi {len(errors)}')


[1/20] gặp/0 -> 20, 21
[2/20] gặp/1 -> 22, 23
[3/20] gặp/10 -> 24
[4/20] gặp/11 -> 25
[5/20] gặp/12 -> 26
[6/20] gặp/13 -> 27
[7/20] gặp/14 -> 28
[8/20] gặp/15 -> 29
[9/20] gặp/16 -> 30
[10/20] gặp/17 -> 31
[11/20] gặp/18 -> 32
[12/20] gặp/19 -> 33
[13/20] gặp/2 -> 34, 35
[14/20] gặp/3 -> 36, 37
[15/20] gặp/4 -> 38, 39
[16/20] gặp/5 -> 40
[17/20] gặp/6 -> 41
[18/20] gặp/7 -> 42
[19/20] gặp/8 -> 43
[20/20] gặp/9 -> 44
Đã tạo 25 mẫu; lỗi 0


## Kiểm tra group


In [12]:
handler = DataHandler()
print('Mẫu đọc được:', handler.validate_dataset())
print('Tổng:', handler.get_class_counts())
print('Group gốc độc lập:', handler.get_origin_counts())
groups = np.asarray(handler.sample_groups('sample'))
augmented = np.asarray([handler.describe(i)['augmented'] for i in range(len(handler))])
assert all((~augmented[groups == group]).any() for group in np.unique(groups))
print(f'{augmented.sum()} bản tăng cường; mọi group đều có mẫu gốc')
print('Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.')


Mẫu đọc được: 1395
Tổng: {'a': 45, 'age': 45, 'b': 45, 'c': 45, 'cảm ơn': 45, 'dấu huyền': 45, 'dấu hỏi': 45, 'dấu nặng': 45, 'dấu sắc': 45, 'dấu â,ô,ê': 45, 'dấu ă': 45, 'dấu ơ,ư': 45, 'e': 45, 'g': 45, 'gặp': 45, 'h': 45, 'i': 45, 'k': 45, 'l': 45, 'm': 45, 'n': 45, 'o': 45, 'r': 45, 't': 45, 'ten_la': 45, 'toi': 45, 'u': 45, 'v': 45, 'xin chào': 45, 'y': 45, 'địa chỉ': 45}
Group gốc độc lập: {'a': 20, 'age': 15, 'b': 5, 'c': 20, 'cảm ơn': 15, 'dấu huyền': 5, 'dấu hỏi': 20, 'dấu nặng': 15, 'dấu sắc': 5, 'dấu â,ô,ê': 5, 'dấu ă': 20, 'dấu ơ,ư': 5, 'e': 5, 'g': 15, 'gặp': 20, 'h': 30, 'i': 15, 'k': 15, 'l': 5, 'm': 30, 'n': 15, 'o': 5, 'r': 15, 't': 15, 'ten_la': 15, 'toi': 5, 'u': 15, 'v': 5, 'xin chào': 15, 'y': 16, 'địa chỉ': 15}
974 bản tăng cường; mọi group đều có mẫu gốc
Chạy lại train.ipynb từ ô đầu để dùng mẫu mới.
